In [ ]:
# ================================================================
# EXERCISE 6
# BIDIRECTIONAL ASSOCIATIVE MEMORY (BAM)
# Handwritten Digit and Code Association System
# ================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split


# ================================================================
# SETTINGS
# ================================================================

# Change this if your CSV has a different name
DATA_FILE = "optdigits.csv"

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)


# ================================================================
# BAM PARAMETERS
# ================================================================

# 10 digits
DIGITS = list(range(10))

# Each digit gets a unique bipolar 10-bit code.
#
# Example:
# 0 -> +1 -1 -1 -1 -1 -1 -1 -1 -1 -1
# 1 -> -1 +1 -1 -1 -1 -1 -1 -1 -1 -1
# ...
#
# This is a one-hot bipolar code.
# It gives every digit a unique code.
#
# X = 64 elements
# Y = 10 elements
# ================================================================


def create_digit_codes():

    codes = {}

    for digit in DIGITS:

        code = -np.ones(
            10,
            dtype=int
        )

        code[digit] = 1

        codes[digit] = code

    return codes


DIGIT_CODES = create_digit_codes()


# ================================================================
# LOAD DATASET
# ================================================================

def load_dataset():

    print("\nLoading Optical Recognition of Handwritten Digits dataset...")

    # ------------------------------------------------------------
    # Try user CSV first
    # ------------------------------------------------------------

    if os.path.exists(DATA_FILE):

        print(
            "Dataset file found:",
            DATA_FILE
        )

        df = pd.read_csv(
            DATA_FILE,
            header=None
        )

        print(
            "CSV Shape:",
            df.shape
        )

        # --------------------------------------------------------
        # UCI optical digits CSV normally has:
        #
        # 64 pixel features + 1 class label
        # --------------------------------------------------------

        if df.shape[1] >= 65:

            # Last column = digit
            X = df.iloc[:, :64].values

            y = df.iloc[:, 64].values

        else:

            raise ValueError(
                "CSV must contain at least "
                "64 pixel columns + 1 label column."
            )

    else:

        # --------------------------------------------------------
        # Backup:
        # sklearn digits dataset
        #
        # This is also an 8x8 handwritten digit dataset.
        # --------------------------------------------------------

        print(
            "\nCSV file not found."
        )

        print(
            "Using sklearn 8x8 handwritten digit dataset "
            "as backup."
        )

        digits_data = load_digits()

        X = digits_data.data

        y = digits_data.target


    # ------------------------------------------------------------
    # Convert labels to integers
    # ------------------------------------------------------------

    y = np.asarray(
        y,
        dtype=int
    )

    # ------------------------------------------------------------
    # Keep only digits 0-9
    # ------------------------------------------------------------

    valid = np.isin(
        y,
        DIGITS
    )

    X = X[valid]
    y = y[valid]


    # ------------------------------------------------------------
    # Normalize original pixels
    # ------------------------------------------------------------

    # Optical digits normally have pixel values 0-16.
    # Some datasets may use 0-255.
    #
    # For BAM we don't need floating point values.
    # We convert them directly to bipolar.
    # ------------------------------------------------------------

    print(
        "\nDataset loaded successfully."
    )

    print(
        "Samples:",
        len(X)
    )

    print(
        "Features:",
        X.shape[1]
    )

    print(
        "Classes:",
        sorted(
            np.unique(y)
        )
    )

    return X, y


X_DATA, Y_DATA = load_dataset()


# ================================================================
# CONVERT 8x8 IMAGE TO BIPOLAR
# ================================================================

def to_bipolar(
    pattern
):

    pattern = np.asarray(
        pattern
    )

    # Pixel > 0 becomes +1
    # Pixel == 0 becomes -1

    return np.where(
        pattern > 0,
        1,
        -1
    )


# ================================================================
# BIPOLAR TO IMAGE
# ================================================================

def bipolar_to_image(
    pattern
):

    pattern = np.asarray(
        pattern
    )

    # +1 -> 1
    # -1 -> 0

    return np.where(
        pattern == 1,
        1,
        0
    ).reshape(
        8,
        8
    )


# ================================================================
# DISPLAY DIGIT
# ================================================================

def display_digit(
    pattern,
    title=""
):

    image = bipolar_to_image(
        pattern
    )

    plt.figure(
        figsize=(3, 3)
    )

    plt.imshow(
        image,
        cmap="gray_r"
    )

    plt.title(
        title
    )

    plt.axis(
        "off"
    )

    plt.show()


# ================================================================
# GET REPRESENTATIVE SAMPLE FOR EACH DIGIT
# ================================================================

def get_representative_samples(
    X,
    y,
    digits_to_use=None
):

    if digits_to_use is None:

        digits_to_use = DIGITS

    samples = {}

    for digit in digits_to_use:

        indexes = np.where(
            y == digit
        )[0]

        if len(indexes) == 0:

            continue

        # First sample
        index = indexes[0]

        samples[digit] = to_bipolar(
            X[index]
        )

    return samples


# ================================================================
# BAM CLASS
# ================================================================

class BAM:

    def __init__(
        self
    ):

        self.W = None

        self.X_patterns = {}

        self.Y_patterns = {}


    # ------------------------------------------------------------
    # Store associations
    # ------------------------------------------------------------

    def train(
        self,
        associations
    ):

        self.X_patterns = {}
        self.Y_patterns = {}

        self.W = np.zeros(
            (
                64,
                10
            ),
            dtype=float
        )

        # --------------------------------------------------------
        # BAM correlation learning rule
        #
        # W = sum(X transpose * Y)
        # --------------------------------------------------------

        for digit, x_pattern in associations.items():

            y_pattern = DIGIT_CODES[
                digit
            ]

            x_pattern = np.asarray(
                x_pattern
            )

            y_pattern = np.asarray(
                y_pattern
            )

            self.W += np.outer(
                x_pattern,
                y_pattern
            )

            self.X_patterns[
                digit
            ] = x_pattern.copy()

            self.Y_patterns[
                digit
            ] = y_pattern.copy()


    # ------------------------------------------------------------
    # Sign function
    # ------------------------------------------------------------

    @staticmethod
    def sign(
        values
    ):

        return np.where(
            values >= 0,
            1,
            -1
        )


    # ------------------------------------------------------------
    # Forward recall
    # X -> Y
    # ------------------------------------------------------------

    def forward(
        self,
        x
    ):

        x = np.asarray(
            x
        )

        y = self.sign(
            np.dot(
                x,
                self.W
            )
        )

        return y


    # ------------------------------------------------------------
    # Backward recall
    # Y -> X
    # ------------------------------------------------------------

    def backward(
        self,
        y
    ):

        y = np.asarray(
            y
        )

        x = self.sign(
            np.dot(
                self.W,
                y
            )
        )

        return x


    # ------------------------------------------------------------
    # Bidirectional stable recall
    # ------------------------------------------------------------

    def stable_recall(
        self,
        x=None,
        y=None,
        max_updates=20
    ):

        if x is not None:

            current_x = np.asarray(
                x
            ).copy()

            current_y = None

        elif y is not None:

            current_y = np.asarray(
                y
            ).copy()

            current_x = None

        else:

            raise ValueError(
                "Either x or y must be provided."
            )


        previous_x = None
        previous_y = None

        for update in range(
            1,
            max_updates + 1
        ):

            # ----------------------------------------------------
            # Forward
            # ----------------------------------------------------

            if current_x is not None:

                current_y = self.forward(
                    current_x
                )

            # ----------------------------------------------------
            # Backward
            # ----------------------------------------------------

            current_x = self.backward(
                current_y
            )

            # ----------------------------------------------------
            # Check stability
            # ----------------------------------------------------

            if (
                previous_x is not None
                and
                previous_y is not None
                and
                np.array_equal(
                    current_x,
                    previous_x
                )
                and
                np.array_equal(
                    current_y,
                    previous_y
                )
            ):

                return (
                    current_x,
                    current_y,
                    update
                )

            previous_x = current_x.copy()
            previous_y = current_y.copy()

        return (
            current_x,
            current_y,
            max_updates
        )


# ================================================================
# GLOBAL BAM
# ================================================================

bam = BAM()

REPRESENTATIVE_PATTERNS = (
    get_representative_samples(
        X_DATA,
        Y_DATA
    )
)


# ================================================================
# 1. STORE DIGIT-CODE PATTERN ASSOCIATIONS
# ================================================================

def store_associations():

    print("\n")
    print("=" * 75)
    print("1. STORE DIGIT-CODE PATTERN ASSOCIATIONS")
    print("=" * 75)


    bam.train(
        REPRESENTATIVE_PATTERNS
    )


    print(
        "\nStored Associations:"
    )

    for digit in sorted(
        bam.X_patterns.keys()
    ):

        print(
            f"Digit {digit} -> Code:",
            DIGIT_CODES[digit]
        )


    print(
        "\nForward Weight Matrix W Shape:",
        bam.W.shape
    )

    print(
        "\nForward Weight Matrix W:"
    )

    print(
        bam.W.astype(int)
    )


    print(
        "\nTranspose Weight Matrix W^T Shape:",
        bam.W.T.shape
    )

    print(
        "\nTranspose Weight Matrix W^T:"
    )

    print(
        bam.W.T.astype(int)
    )


# ================================================================
# 2. FORWARD DIGIT RECALL CHALLENGE
# ================================================================

def forward_recall():

    print("\n")
    print("=" * 75)
    print("2. FORWARD DIGIT RECALL CHALLENGE")
    print("=" * 75)


    if bam.W is None:

        print(
            "\nBAM is not trained."
        )

        print(
            "Please select option 1 first."
        )

        return


    digit = int(
        input(
            "\nEnter digit to test (0-9): "
        )
    )


    if digit not in bam.X_patterns:

        print(
            "Digit not available."
        )

        return


    x = bam.X_patterns[
        digit
    ]


    y = bam.forward(
        x
    )


    expected_code = DIGIT_CODES[
        digit
    ]


    # ------------------------------------------------------------
    # Identify digit from code
    # ------------------------------------------------------------

    recalled_digit = decode_digit(
        y
    )


    print(
        "\nOriginal Digit:",
        digit
    )

    print(
        "\nInput 8x8 Digit Pattern:"
    )

    print(
        bipolar_to_image(
            x
        )
    )


    print(
        "\nCalculated Y-Layer State:"
    )

    print(
        y
    )


    print(
        "\nExpected Code:"
    )

    print(
        expected_code
    )


    print(
        "\nStable Output Code:"
    )

    print(
        y
    )


    print(
        "\nIdentified Digit:",
        recalled_digit
    )


    if recalled_digit == digit:

        print(
            "\nForward Recall: CORRECT"
        )

    else:

        print(
            "\nForward Recall: INCORRECT"
        )


# ================================================================
# DECODE BIPOLAR DIGIT CODE
# ================================================================

def decode_digit(
    code
):

    code = np.asarray(
        code
    )

    best_digit = None

    best_score = -999999

    for digit in DIGITS:

        expected = DIGIT_CODES[
            digit
        ]

        score = np.sum(
            code == expected
        )

        if score > best_score:

            best_score = score

            best_digit = digit

    return best_digit


# ================================================================
# 3. REVERSE DIGIT RECONSTRUCTION
# ================================================================

def reverse_reconstruction():

    print("\n")
    print("=" * 75)
    print("3. REVERSE DIGIT RECONSTRUCTION CHALLENGE")
    print("=" * 75)


    if bam.W is None:

        print(
            "\nPlease train BAM first."
        )

        return


    digit = int(
        input(
            "\nEnter digit code to reconstruct (0-9): "
        )
    )


    if digit not in bam.Y_patterns:

        print(
            "Digit not available."
        )

        return


    y = DIGIT_CODES[
        digit
    ]


    reconstructed_x = bam.backward(
        y
    )


    original_x = bam.X_patterns[
        digit
    ]


    accuracy = np.mean(
        reconstructed_x
        ==
        original_x
    ) * 100


    print(
        "\nInput Code:"
    )

    print(
        y
    )


    print(
        "\nReconstructed Digit:"
    )

    print(
        bipolar_to_image(
            reconstructed_x
        )
    )


    print(
        "\nOriginal Digit:"
    )

    print(
        bipolar_to_image(
            original_x
        )
    )


    print(
        "\nReconstruction Accuracy:",
        round(
            accuracy,
            2
        ),
        "%"
    )


    display_digit(
        reconstructed_x,
        f"Reconstructed Digit {digit}"
    )


# ================================================================
# 4. NOISY PATTERN ASSOCIATION TEST
# ================================================================

def noisy_pattern_test():

    print("\n")
    print("=" * 75)
    print("4. NOISY PATTERN ASSOCIATION TEST")
    print("=" * 75)


    if bam.W is None:

        print(
            "\nPlease train BAM first."
        )

        return


    digit = int(
        input(
            "\nEnter digit to test: "
        )
    )


    if digit not in bam.X_patterns:

        print(
            "Digit not available."
        )

        return


    original = bam.X_patterns[
        digit
    ]


    noise_levels = [
        5,
        10,
        20,
        30,
        40
    ]


    print(
        "\nNoise Analysis:"
    )


    maximum_tolerated = 0


    for noise in noise_levels:

        noisy = original.copy()


        number_to_flip = int(
            64 * noise / 100
        )


        indexes = np.random.choice(
            64,
            number_to_flip,
            replace=False
        )


        noisy[indexes] *= -1


        recalled_code = bam.forward(
            noisy
        )


        recalled_digit = decode_digit(
            recalled_code
        )


        correct = (
            recalled_digit == digit
        )


        if correct:

            maximum_tolerated = noise


        print(
            "\n"
            + "-" * 50
        )

        print(
            "Corruption:",
            noise,
            "%"
        )


        print(
            "Noisy Digit:"
        )

        print(
            bipolar_to_image(
                noisy
            )
        )


        print(
            "Recalled Code:",
            recalled_code
        )


        print(
            "Identified Digit:",
            recalled_digit
        )


        print(
            "Correct:",
            correct
        )


    print(
        "\nMaximum Noise Level Tolerated:",
        maximum_tolerated,
        "%"
    )


# ================================================================
# 5. PARTIAL DIGIT COMPLETION CHALLENGE
# ================================================================

def partial_digit_completion():

    print("\n")
    print("=" * 75)
    print("5. PARTIAL DIGIT COMPLETION CHALLENGE")
    print("=" * 75)


    if bam.W is None:

        print(
            "\nPlease train BAM first."
        )

        return


    digit = int(
        input(
            "\nEnter digit: "
        )
    )


    if digit not in bam.X_patterns:

        print(
            "Digit not available."
        )

        return


    original = bam.X_patterns[
        digit
    ].copy()


    partial = original.copy()


    print(
        "\nChoose missing region:"
    )

    print(
        "1. Remove rows"
    )

    print(
        "2. Remove columns"
    )

    print(
        "3. Remove center rectangle"
    )


    choice = input(
        "\nEnter choice: "
    )


    image = partial.reshape(
        8,
        8
    )


    if choice == "1":

        # Remove rows 3 and 4

        image[
            3:5,
            :
        ] = -1


    elif choice == "2":

        # Remove columns 3 and 4

        image[
            :,
            3:5
        ] = -1


    elif choice == "3":

        # Remove center rectangle

        image[
            2:6,
            2:6
        ] = -1


    else:

        print(
            "Invalid choice."
        )

        return


    partial = image.flatten()


    print(
        "\nPartial Input:"
    )

    print(
        image
    )


    reconstructed_x, reconstructed_y, updates = (
        bam.stable_recall(
            x=partial
        )
    )


    accuracy = np.mean(
        reconstructed_x
        ==
        original
    ) * 100


    print(
        "\nFinal Reconstructed Digit:"
    )

    print(
        bipolar_to_image(
            reconstructed_x
        )
    )


    print(
        "\nReconstructed Code:"
    )

    print(
        reconstructed_y
    )


    print(
        "\nState Updates:",
        updates
    )


    print(
        "\nCorrectly Reconstructed Pixels:",
        round(
            accuracy,
            2
        ),
        "%"
    )


# ================================================================
# 6. BIDIRECTIONAL RECALL CONSISTENCY ANALYZER
# ================================================================

def consistency_analyzer():

    print("\n")
    print("=" * 75)
    print("6. BIDIRECTIONAL RECALL CONSISTENCY ANALYZER")
    print("=" * 75)


    if bam.W is None:

        print(
            "\nPlease train BAM first."
        )

        return


    results = []


    for digit in sorted(
        bam.X_patterns.keys()
    ):

        original_x = bam.X_patterns[
            digit
        ]


        # X -> Y
        recalled_y = bam.forward(
            original_x
        )


        # Y -> X
        reconstructed_x = bam.backward(
            recalled_y
        )


        consistency = np.mean(
            reconstructed_x
            ==
            original_x
        ) * 100


        recalled_digit = decode_digit(
            recalled_y
        )


        correct = (
            recalled_digit == digit
        )


        results.append(
            {
                "Digit": digit,
                "Recalled Digit": recalled_digit,
                "Forward Correct": correct,
                "Consistency %": consistency
            }
        )


    result_df = pd.DataFrame(
        results
    )


    print(
        "\nBidirectional Consistency:"
    )

    print(
        result_df.round(2).to_string(
            index=False
        )
    )


    average = result_df[
        "Consistency %"
    ].mean()


    print(
        "\nAverage Bidirectional Consistency:",
        round(
            average,
            2
        ),
        "%"
    )


    failures = result_df[
        result_df["Consistency %"] < 100
    ]


    if len(failures) > 0:

        print(
            "\nAssociations failing perfect consistency:"
        )

        print(
            failures.to_string(
                index=False
            )
        )

    else:

        print(
            "\nAll stored associations "
            "returned to the original pattern."
        )


# ================================================================
# 7. ASSOCIATION CAPACITY CHALLENGE
# ================================================================

def capacity_challenge():

    print("\n")
    print("=" * 75)
    print("7. ASSOCIATION CAPACITY CHALLENGE")
    print("=" * 75)


    association_counts = [
        2,
        4,
        6,
        8,
        10
    ]


    results = []


    for count in association_counts:

        selected_digits = DIGITS[
            :count
        ]


        associations = {
            digit:
            REPRESENTATIVE_PATTERNS[digit]

            for digit in selected_digits

            if digit in REPRESENTATIVE_PATTERNS
        }


        test_bam = BAM()


        test_bam.train(
            associations
        )


        # --------------------------------------------------------
        # Forward accuracy
        # --------------------------------------------------------

        forward_correct = 0


        for digit, x in associations.items():

            y = test_bam.forward(
                x
            )


            recalled_digit = decode_digit(
                y
            )


            if recalled_digit == digit:

                forward_correct += 1


        forward_accuracy = (
            forward_correct
            /
            len(associations)
            *
            100
        )


        # --------------------------------------------------------
        # Backward reconstruction accuracy
        # --------------------------------------------------------

        reconstruction_scores = []


        for digit, x in associations.items():

            y = DIGIT_CODES[
                digit
            ]


            reconstructed = test_bam.backward(
                y
            )


            score = np.mean(
                reconstructed == x
            ) * 100


            reconstruction_scores.append(
                score
            )


        backward_accuracy = np.mean(
            reconstruction_scores
        )


        # --------------------------------------------------------
        # Bidirectional consistency
        # --------------------------------------------------------

        consistency_scores = []


        for digit, x in associations.items():

            y = test_bam.forward(
                x
            )


            reconstructed = test_bam.backward(
                y
            )


            score = np.mean(
                reconstructed == x
            ) * 100


            consistency_scores.append(
                score
            )


        consistency = np.mean(
            consistency_scores
        )


        results.append(
            {
                "Associations": count,
                "Forward Accuracy %":
                    forward_accuracy,
                "Backward Accuracy %":
                    backward_accuracy,
                "Consistency %":
                    consistency
            }
        )


    result_df = pd.DataFrame(
        results
    )


    print(
        "\nCapacity Analysis:"
    )

    print(
        result_df.round(2).to_string(
            index=False
        )
    )


    # ------------------------------------------------------------
    # Find performance decrease
    # ------------------------------------------------------------

    best_accuracy = (
        result_df[
            "Forward Accuracy %"
        ].max()
    )


    decrease_point = None


    for i in range(
        1,
        len(result_df)
    ):

        if (
            result_df.loc[
                i,
                "Forward Accuracy %"
            ]
            <
            result_df.loc[
                i - 1,
                "Forward Accuracy %"
            ]
        ):

            decrease_point = (
                result_df.loc[
                    i,
                    "Associations"
                ]
            )

            break


    if decrease_point is not None:

        print(
            "\nRecall performance begins "
            "to decrease after approximately",
            decrease_point,
            "associations."
        )

    else:

        print(
            "\nNo decrease detected within "
            "the tested association counts."
        )


    # ------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------

    plt.figure(
        figsize=(9, 5)
    )


    plt.plot(
        result_df["Associations"],
        result_df["Forward Accuracy %"],
        marker="o",
        label="Forward"
    )


    plt.plot(
        result_df["Associations"],
        result_df["Backward Accuracy %"],
        marker="o",
        label="Backward"
    )


    plt.plot(
        result_df["Associations"],
        result_df["Consistency %"],
        marker="o",
        label="Consistency"
    )


    plt.xlabel(
        "Number of Associations"
    )

    plt.ylabel(
        "Accuracy (%)"
    )

    plt.title(
        "BAM Association Capacity"
    )

    plt.legend()

    plt.grid(
        True
    )

    plt.show()


# ================================================================
# 8. CROSS-ASSOCIATION ERROR DETECTOR
# ================================================================

def cross_association_detector():

    print("\n")
    print("=" * 75)
    print("8. CROSS-ASSOCIATION ERROR DETECTOR")
    print("=" * 75)


    if bam.W is None:

        print(
            "\nPlease train BAM first."
        )

        return


    error_matrix = np.zeros(
        (
            10,
            10
        ),
        dtype=int
    )


    results = []


    for digit in sorted(
        bam.X_patterns.keys()
    ):

        x = bam.X_patterns[
            digit
        ]


        recalled_y = bam.forward(
            x
        )


        recalled_digit = decode_digit(
            recalled_y
        )


        _, _, updates = bam.stable_recall(
            x=x
        )


        error_matrix[
            digit,
            recalled_digit
        ] += 1


        if recalled_digit != digit:

            results.append(
                {
                    "Input Digit": digit,
                    "Expected Code":
                        DIGIT_CODES[digit],
                    "Recalled Code":
                        recalled_y,
                    "Incorrectly Associated Digit":
                        recalled_digit,
                    "State Updates":
                        updates
                }
            )


    # ------------------------------------------------------------
    # Display errors
    # ------------------------------------------------------------

    if len(results) > 0:

        print(
            "\nCross-Association Errors:"
        )


        for result in results:

            print(
                "\nInput Digit:",
                result["Input Digit"]
            )

            print(
                "Expected Code:",
                result["Expected Code"]
            )

            print(
                "Recalled Code:",
                result["Recalled Code"]
            )

            print(
                "Incorrectly Associated Digit:",
                result[
                    "Incorrectly Associated Digit"
                ]
            )

            print(
                "State Updates:",
                result[
                    "State Updates"
                ]
            )

    else:

        print(
            "\nNo cross-association errors detected."
        )


    # ------------------------------------------------------------
    # Error matrix
    # ------------------------------------------------------------

    print(
        "\nDigit-to-Digit Association Matrix:"
    )

    print(
        pd.DataFrame(
            error_matrix,
            index=[
                f"Input {i}"
                for i in DIGITS
            ],
            columns=[
                f"Recall {i}"
                for i in DIGITS
            ]
        )
    )


    # ------------------------------------------------------------
    # Find most frequent confusion
    # ------------------------------------------------------------

    confusion_matrix = (
        error_matrix.copy()
    )


    # Remove correct diagonal
    np.fill_diagonal(
        confusion_matrix,
        0
    )


    if np.max(
        confusion_matrix
    ) > 0:

        index = np.unravel_index(
            np.argmax(
                confusion_matrix
            ),
            confusion_matrix.shape
        )


        print(
            "\nMost Frequently Confused Pair:"
        )

        print(
            f"Digit {index[0]} "
            f"-> Digit {index[1]}"
        )

    else:

        print(
            "\nNo confusion pair exists."
        )


# ================================================================
# 9. BAM ASSOCIATIVE MEMORY REPORT
# ================================================================

def bam_report():

    print("\n")
    print("=" * 75)
    print("9. BAM ASSOCIATIVE MEMORY REPORT")
    print("=" * 75)


    if bam.W is None:

        print(
            "\nTraining BAM first..."
        )

        bam.train(
            REPRESENTATIVE_PATTERNS
        )


    stored_count = len(
        bam.X_patterns
    )


    # ------------------------------------------------------------
    # Forward recall accuracy
    # ------------------------------------------------------------

    forward_correct = 0


    for digit, x in bam.X_patterns.items():

        y = bam.forward(
            x
        )


        recalled_digit = decode_digit(
            y
        )


        if recalled_digit == digit:

            forward_correct += 1


    forward_accuracy = (
        forward_correct
        /
        stored_count
        *
        100
    )


    # ------------------------------------------------------------
    # Backward accuracy
    # ------------------------------------------------------------

    backward_scores = []


    for digit, x in bam.X_patterns.items():

        y = DIGIT_CODES[
            digit
        ]


        reconstructed = bam.backward(
            y
        )


        score = np.mean(
            reconstructed == x
        ) * 100


        backward_scores.append(
            score
        )


    backward_accuracy = np.mean(
        backward_scores
    )


    # ------------------------------------------------------------
    # Bidirectional consistency
    # ------------------------------------------------------------

    consistency_scores = []


    for digit, x in bam.X_patterns.items():

        y = bam.forward(
            x
        )


        reconstructed = bam.backward(
            y
        )


        score = np.mean(
            reconstructed == x
        ) * 100


        consistency_scores.append(
            score
        )


    consistency = np.mean(
        consistency_scores
    )


    # ------------------------------------------------------------
    # Noise tolerance
    # ------------------------------------------------------------

    noise_results = []


    for digit, original in bam.X_patterns.items():

        maximum_noise = 0


        for noise in [
            5,
            10,
            20,
            30,
            40
        ]:

            noisy = original.copy()


            number_to_flip = int(
                64 * noise / 100
            )


            indexes = np.random.choice(
                64,
                number_to_flip,
                replace=False
            )


            noisy[indexes] *= -1


            recalled = bam.forward(
                noisy
            )


            recalled_digit = decode_digit(
                recalled
            )


            if recalled_digit == digit:

                maximum_noise = noise


        noise_results.append(
            maximum_noise
        )


    average_noise = np.mean(
        noise_results
    )


    # ------------------------------------------------------------
    # Cross association errors
    # ------------------------------------------------------------

    errors = 0


    for digit, x in bam.X_patterns.items():

        y = bam.forward(
            x
        )


        recalled_digit = decode_digit(
            y
        )


        if recalled_digit != digit:

            errors += 1


    # ------------------------------------------------------------
    # Print report
    # ------------------------------------------------------------

    print(
        "\nBAM ASSOCIATIVE MEMORY REPORT"
    )

    print(
        "-" * 55
    )

    print(
        "Stored Associations:",
        stored_count
    )

    print(
        "Forward Recall Accuracy:",
        round(
            forward_accuracy,
            2
        ),
        "%"
    )

    print(
        "Backward Reconstruction Accuracy:",
        round(
            backward_accuracy,
            2
        ),
        "%"
    )

    print(
        "Bidirectional Consistency:",
        round(
            consistency,
            2
        ),
        "%"
    )

    print(
        "Average Noise Tolerance:",
        round(
            average_noise,
            2
        ),
        "%"
    )

    print(
        "Cross-Association Errors:",
        errors
    )


    # ------------------------------------------------------------
    # Most robust association
    # ------------------------------------------------------------

    if len(noise_results) > 0:

        robust_index = np.argmax(
            noise_results
        )

        robust_digit = list(
            bam.X_patterns.keys()
        )[robust_index]


        print(
            "\nMost Robust Digit-Code Association:"
        )

        print(
            "Digit:",
            robust_digit
        )

        print(
            "Noise Tolerance:",
            noise_results[
                robust_index
            ],
            "%"
        )


    print(
        "\nMaximum Reliable BAM Capacity:"
    )

    print(
        "See Association Capacity Challenge "
        "(Menu 7) for the measured capacity."
    )


# ================================================================
# 10. EXIT
# ================================================================

def exit_program():

    print(
        "\nProgram terminated successfully."
    )


# ================================================================
# MAIN MENU
# ================================================================

while True:

    print("\n")

    print(
        "=" * 75
    )

    print(
        " BIDIRECTIONAL ASSOCIATIVE MEMORY (BAM)"
    )

    print(
        " HANDWRITTEN DIGIT AND CODE ASSOCIATION SYSTEM"
    )

    print(
        "=" * 75
    )

    print(
        "1. Store Digit-Code Pattern Associations"
    )

    print(
        "2. Forward Digit Recall Challenge"
    )

    print(
        "3. Reverse Digit Reconstruction Challenge"
    )

    print(
        "4. Noisy Pattern Association Test"
    )

    print(
        "5. Partial Digit Completion Challenge"
    )

    print(
        "6. Bidirectional Recall Consistency Analyzer"
    )

    print(
        "7. Association Capacity Challenge"
    )

    print(
        "8. Cross-Association Error Detector"
    )

    print(
        "9. BAM Associative Memory Report"
    )

    print(
        "10. Exit"
    )


    choice = input(
        "\nEnter your choice: "
    )


    if choice == "1":

        store_associations()


    elif choice == "2":

        forward_recall()


    elif choice == "3":

        reverse_reconstruction()


    elif choice == "4":

        noisy_pattern_test()


    elif choice == "5":

        partial_digit_completion()


    elif choice == "6":

        consistency_analyzer()


    elif choice == "7":

        capacity_challenge()


    elif choice == "8":

        cross_association_detector()


    elif choice == "9":

        bam_report()


    elif choice == "10":

        exit_program()

        break


    else:

        print(
            "\nInvalid choice."
        )

        print(
            "Please enter a number from 1 to 10."
        )


Loading Optical Recognition of Handwritten Digits dataset...

CSV file not found.
Using sklearn 8x8 handwritten digit dataset as backup.

Dataset loaded successfully.
Samples: 1797
Features: 64
Classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


 BIDIRECTIONAL ASSOCIATIVE MEMORY (BAM)
 HANDWRITTEN DIGIT AND CODE ASSOCIATION SYSTEM
1. Store Digit-Code Pattern Associations
2. Forward Digit Recall Challenge
3. Reverse Digit Reconstruction Challenge
4. Noisy Pattern Association Test
5. Partial Digit Completion Challenge
6. Bidirectional Recall Consistency Analyzer
7. Association Capacity Challenge
8. Cross-Association Error Detector
9. BAM Associative Memory Report
10. Exit
